# KOPIS CLI 오프라인 검증

실제 API를 호출하지 않고 가짜 응답·가짜 시계를 사용해 1초 간격, HTTP 400 재시도·기타 차단 종료, 중지·재개, 동시 실행 잠금, 인증키 비노출을 검증한다. 검증용 데이터는 `tmp/final_kopis_cli_tests` 아래 임시 폴더에만 만들고 제거한다. 원본 자료와 실제 수집 상태는 수정하지 않는다.

이 노트북은 필요할 때만 사용하는 **수집 관리 도구**이며 최종 분석의 전체 실행에는 포함하지 않는다. 수집 및 스냅샷 등록은 공용 원자료와 `metadata/`에 기록된다. `jupyter-notebook/20261006/metadata/`에 고정한 분석 입력은 자동 교체하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
from contextlib import contextmanager, redirect_stdout
from unittest.mock import patch
import csv, io, json, os, signal, subprocess, sys, tempfile, urllib.error, urllib.parse
import pandas as pd
from IPython.display import display

ROOT = PROJECT_ROOT
NOTEBOOK = project_path('jupyter-notebook/20261006/data-collection/01_KOPIS_상세수집.ipynb')
RUNTIME = project_json_loads(NOTEBOOK.read_text(encoding="utf-8"))
TEST_ROOT = (project_path('tmp/final_kopis_cli_tests')).resolve()
assert TEST_ROOT.is_relative_to(ROOT.resolve())
TEST_ROOT.mkdir(parents=True, exist_ok=True)
TEST_KEY = "SYNTHETIC_TEST_CREDENTIAL_ONLY"
results = []

def load_runtime(root):
    scope = {"KOPIS_PROJECT_ROOT": str(root)}
    for cell in RUNTIME["cells"]:
        if cell["cell_type"] == "code" and "kopis-runtime" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), str(NOTEBOOK), "exec"), scope)
    return scope

class FakeClock:
    value = 100.0
    def monotonic(self): return self.value
    def sleep(self, seconds): self.value += max(seconds, 1e-9)

class FakeResponse:
    status = 200
    def __init__(self, data): self.data = data
    def read(self): return self.data
    def __enter__(self): return self
    def __exit__(self, *args): return False

## 1. 수집 격리와 응답 모형

실제 요청 함수의 URL 구성·HTTP 오류 처리·XML ID 검증·저장을 실행하되, 네트워크 전송 부분만 가짜 응답으로 바꾼다. 예외 URL에는 가짜 인증키를 넣어 오류 처리 중 키가 유출되지 않는지도 검사한다.

In [ ]:
class FakeOpener:
    def __init__(self, scope, clock, blocked_at=None, stop_at=None, interrupt_at=None, blocked_status=400):
        self.scope, self.clock = scope, clock
        self.blocked_at = {blocked_at} if isinstance(blocked_at, int) else set(blocked_at or [])
        self.stop_at, self.interrupt_at, self.blocked_status = stop_at, interrupt_at, blocked_status
        self.calls = []
    def open(self, request, timeout):
        url = urllib.parse.urlsplit(request.full_url)
        record_id = url.path.rsplit("/", 1)[1]
        self.calls.append((self.clock.monotonic(), record_id))
        self.clock.sleep(0.07)
        n = len(self.calls)
        if n in self.blocked_at:
            raise urllib.error.HTTPError(request.full_url, self.blocked_status, "blocked", {}, io.BytesIO(b"Request Blocked"))
        if n == self.interrupt_at:
            raise KeyboardInterrupt()
        idkey = "mt10id" if record_id.startswith("FC") else "mt20id"
        xml = f"<dbs><db><{idkey}>{record_id}</{idkey}><name>fixture</name></db></dbs>".encode()
        if n == self.stop_at:
            (self.scope["REPORTS"] / "STOP").touch()
        return FakeResponse(xml)

@contextmanager
def scenario(**options):
    with tempfile.TemporaryDirectory(prefix="case_", dir=str(TEST_ROOT)) as directory:
        root = Path(directory).resolve()
        assert root.is_relative_to(TEST_ROOT)
        (root / "AGENTS.md").write_text("Offline test fixture", encoding="utf-8")
        legacy = root / "dataset/processed/kopis"
        legacy.mkdir(parents=True)
        (legacy / "facilities_list_seoul.csv").write_text("mt10id,gugunnm\nFC1,종로구\nFC2,마포구\n", encoding="utf-8-sig")
        (legacy / "performance_detail_candidates.csv").write_text("mt20id\nPF1\nPF2\n", encoding="utf-8-sig")
        scope, clock = load_runtime(root), FakeClock()
        scope["time"] = clock
        scope["load_key"] = lambda: TEST_KEY
        opener, log = FakeOpener(scope, clock, **options), io.StringIO()
        with patch("urllib.request.build_opener", return_value=opener), redirect_stdout(log):
            yield scope, opener, root, log
        assert TEST_KEY not in log.getvalue()
        assert all(TEST_KEY.encode() not in p.read_bytes() for p in root.rglob("*") if p.is_file())

def state(scope):
    return project_json_loads((scope["REPORTS"] / "latest.json").read_text(encoding="utf-8"))

def record(name): results.append({"검증": name, "결과": "통과"})

## 2. 완료·간격·재개·요청 건수 제한

완료한 ID는 재실행해도 호출하지 않는다. 1초보다 짧게 호출되는 구간이 없는지 가짜 전송 시각으로 확인한다.

In [ ]:
with scenario() as (scope, api, root, log):
    assert scope["collect"]() == 0
    first = state(scope)
    assert first["status"] == "completed_pending_queue" and len(api.calls) == 4
    assert first["request_interval_seconds"] == 1.0
    assert all(abs((b[0]-a[0]) - 1.07) < 1e-8 for a,b in zip(api.calls, api.calls[1:]))
    assert first["pending_counts"] == {"facility": 0, "performance": 0}
    assert scope["collect"]() == 0 and len(api.calls) == 4
record("1초 대기 + 응답 시간·전체 완료·완료 ID 중복 호출 방지")

with scenario() as (scope, api, root, log):
    assert scope["collect"](limit=1) == 0
    assert state(scope)["status"] == "request_limit_reached" and len(api.calls) == 1
record("--limit 1에서 한 건만 요청")

## 3. 차단·사용자 중지

HTTP 400은 현재 ID를 3초 뒤 다시 요청하고, HTTP 401/403/429는 후속 요청 없이 중단한다. 연속 400과 재시도 대기 중 사용자 중지를 검사한다. 중지 후 재개하면 이전 정상 원문은 재사용한다.

In [ ]:
with scenario(blocked_at={1, 2}) as (scope, api, root, log):
    assert scope["collect"]() == 0
    completed = state(scope)
    assert len(api.calls) == 6 and completed["status"] == "completed_pending_queue"
    assert completed["http400_responses"] == completed["http400_retry_count"] == 2
    assert completed["http_requests_sent"] == 6 and completed["requests_attempted"] == 4
    assert len({record_id for _,record_id in api.calls[:3]}) == 1
    assert all(abs((b[0]-a[0]) - 3.07) < 1e-8 for a,b in zip(api.calls[:2], api.calls[1:3]))
    assert len(list((root / "dataset/raw/kopis_recollection").rglob("*.xml"))) == 4
    assert completed["next_retry_at_kst"] is None and completed["retrying_id"] is None
record("연속 HTTP 400 후 같은 ID를 3초 뒤 재시도·응답/재시도/ID 수 분리·중복 저장 없음")

with scenario(blocked_at={1, 2}) as (scope, api, root, log):
    assert scope["collect"](limit=1) == 0
    assert len(api.calls) == 3 and state(scope)["requests_attempted"] == 1
    assert state(scope)["status"] == "request_limit_reached"
record("--limit은 재시도를 포함한 HTTP 수가 아닌 처리 ID 수를 제한")

for status_code in (401, 403, 429):
    with scenario(blocked_at=2, blocked_status=status_code) as (scope, api, root, log):
        assert scope["collect"]() == 3
        stopped = state(scope)
        assert len(api.calls) == 2 and stopped["status"] == "api_access_rejected"
        assert stopped["blocked_marker"] and stopped["http_status"] == status_code
        assert stopped["http400_retry_count"] == 0
        assert sum(stopped["pending_counts"].values()) == 3
        assert stopped["estimated_remaining_seconds"] is None
        assert scope["kopis_cli_main"](["status"]) == 0
        assert scope["kopis_cli_main"](["watch"]) == 0
        assert scope["kopis_cli_main"](["stop"]) == 0
        assert len(api.calls) == 2
record("HTTP 401/403/429 즉시 중단·현황 명령은 오프라인")

for stop_mode in ("stop_file", "ctrl_c"):
    with scenario(blocked_at=1) as (scope, api, root, log):
        original_sleep = scope["time"].sleep
        def stop_during_wait(seconds):
            original_sleep(seconds)
            if state(scope)["status"] == "retry_wait":
                if stop_mode == "stop_file":
                    (scope["REPORTS"] / "STOP").touch()
                else:
                    signal.getsignal(signal.SIGINT)(signal.SIGINT, None)
        scope["time"].sleep = stop_during_wait
        assert scope["collect"]() == 0 and len(api.calls) == 1
        stopped = state(scope)
        assert stopped["status"] == "stopped_by_user"
        assert stopped["http400_responses"] == 1 and stopped["http400_retry_count"] == 0
        assert stopped["elapsed_seconds"] < 3 and not scope["collector_active"]()
record("3초 재시도 대기 중 stop 파일·Ctrl+C 즉시 반응·추가 요청 없음")

if os.name == "nt":
    with scenario(blocked_at=1) as (scope, api, root, log):
        with patch("sys.stdin.isatty", return_value=True), \
             patch("msvcrt.kbhit", side_effect=lambda: state(scope)["status"] == "retry_wait"), \
             patch("msvcrt.getwch", return_value="q"):
            assert scope["collect"]() == 0 and len(api.calls) == 1
        assert state(scope)["status"] == "stopped_by_user"
    record("재시도 대기 중 Q 키로 중지")

with scenario(stop_at=1) as (scope, api, root, log):
    assert scope["collect"]() == 0 and len(api.calls) == 1
    assert state(scope)["status"] == "stopped_by_user"
    assert scope["collect"]() == 0 and len(api.calls) == 4
    assert len({record_id for _,record_id in api.calls}) == 4
record("중지 파일로 종료·재개 시 저장된 ID 재사용")

with scenario(interrupt_at=2) as (scope, api, root, log):
    assert scope["collect"]() == 0
    assert state(scope)["status"] == "stopped_by_user" and len(api.calls) == 2
    assert not scope["collector_active"]()
record("키보드 중단 후 진행 저장·잠금 해제")

## 4. 별도 프로세스 중복 실행 차단

첫 프로세스가 잠금을 가진 동안 별도 Python 프로세스에서 `run`을 시도한다. 두 번째 실행은 API를 호출하기 전에 거부되어야 한다.

In [ ]:
with scenario() as (scope, api, root, log):
    with scope["collector_lock"]() as acquired:
        assert acquired and scope["collector_active"]()
        assert scope["request_stop"]() == 0 and (scope["REPORTS"] / "STOP").exists()
        child = """import json,sys
from pathlib import Path
n=json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
s={'KOPIS_PROJECT_ROOT':sys.argv[2]}
for c in n['cells']:
    if c['cell_type']=='code' and 'kopis-runtime' in c.get('metadata',{}).get('tags',[]):
        exec(''.join(c['source']),s)
def forbidden(): raise AssertionError('key/network must not be reached')
s['load_key']=forbidden
sys.exit(s['kopis_cli_main'](['run']))
"""
        process = subprocess.run([sys.executable, "-c", child, str(NOTEBOOK), str(root)],
                                 capture_output=True, timeout=20,
                                 creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0)
        assert process.returncode == 2 and not api.calls
    assert not scope["collector_active"]()
record("별도 프로세스 동시 수집 거부·stop 명령 기록·잠금 해제")
record("가짜 인증키가 응답 파일·상태·출력에 남지 않음")
display(pd.DataFrame(results))
(project_path('reports/20261006/cli_offline_validation.json')).write_text(
    json.dumps({"network_requests": 0, "checks": results}, ensure_ascii=False, indent=2), encoding="utf-8")

해석: 검증은 CLI의 제어 흐름과 저장 동작에 대한 결과이며 실제 서버의 차단 해제를 확인하지 않는다. 최신 실제 API 실행은 기존 수집 기록을 따른다.